installations 

In [1]:
!pip install -q -U transformers datasets bitsandbytes accelerate torch huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 90.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 46.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 3.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 10.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 8.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 30.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 9.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 4.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━

load model

In [7]:
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. Retrieve the token from Kaggle Secrets and Login
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("IRProject")
login(token=hf_token)

# 2. Configure 4-bit quantization to fit the model into VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 3. Load MedGemma 4B Instruction-Tuned
model_id = "google/medgemma-4b-pt" # Using the base/pt or it variant
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model weights... this may take a few minutes.")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)
print("Model loaded successfully!")

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/medgemma-4b-pt.
403 Client Error. (Request ID: Root=1-6a783856-1853e295658974b427ac3f5c;91b6a43e-9718-4361-b775-5e71ab34951d)

Cannot access gated repo for url https://huggingface.co/google/medgemma-4b-pt/resolve/main/config.json.
Access to model google/medgemma-4b-pt is restricted and you are not in the authorized list. Visit https://huggingface.co/google/medgemma-4b-pt to ask for access.

dataset

In [ ]:
from datasets import load_dataset

# Load the reasoning hallucination test subset from Med-HALT
dataset = load_dataset("openlifescienceai/Med-HALT", "reasoning_fake", split="test")

# Take a small sample of 50 examples for rapid prototyping
test_sample = dataset.select(range(50))
print(f"Loaded {len(test_sample)} medical test cases.")

baseline

In [ ]:
def evaluate_prompt(model, tokenizer, prompt_text):
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=100, 
            temperature=0.1 # Low temperature keeps it deterministic for experiments
        )
        
    # Decode the output, ignoring the prompt itself
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    return response

failure_cases = []
print("Running Baseline Evaluation...")

for item in test_sample:
    question = item['question']
    expected_answer = item['correct_answer']
    
    prompt = f"Medical Question: {question}\nAnswer clearly and accurately:"
    model_response = evaluate_prompt(model, tokenizer, prompt)
    
    # Simple string matching to check if it got the right fact
    # In an advanced lab, you might use an LLM-as-a-judge here instead
    if expected_answer.lower() not in model_response.lower():
        failure_cases.append({
            "question": question,
            "expected": expected_answer,
            "baseline_response": model_response
        })

print(f"Baseline Complete: Found {len(failure_cases)} definitive hallucinations/failures.")

architectural experiment 

In [ ]:
import torch.nn as nn

# Define your new, experimental math function
class MyExperimentalGFunction(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, x):
        # Example: A custom modification of the SiLU function
        return x * torch.sigmoid(x) + (0.05 * x)

print("Injecting custom architecture...")

# Iterate through the model layers and replace standard activation functions
# Note: Gemma models usually use 'act_fn' inside their MLP blocks
for name, module in model.named_modules():
    if hasattr(module, 'act_fn'): 
        module.act_fn = MyExperimentalGFunction()
        
print("Architecture modified successfully!")

reevaluate 

In [ ]:
print("Re-evaluating failure cases with modified architecture...")

fixed_cases = 0
for case in failure_cases:
    prompt = f"Medical Question: {case['question']}\nAnswer clearly and accurately:"
    new_response = evaluate_prompt(model, tokenizer, prompt)
    
    if case['expected'].lower() in new_response.lower():
        print(f"FIXED: {case['question']}")
        fixed_cases += 1
    else:
        # It still failed, but you can inspect `new_response` to see HOW it failed differently
        pass

print(f"Lab Results: Your architectural change fixed {fixed_cases} out of {len(failure_cases)} hallucinations.")